In [2]:
import sys
from pathlib import Path


PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )

print(f"Project root: {PROJECT_ROOT}")

Project root: c:\Users\jsteve\OneDrive - Burlington\Documents\Python Scripts\Projects\Data-Science-Projects\medicare-claims-intelligence-agent


In [ ]:
from agents.router import route_question

# Checking for population tool
route_question(
    "What percentage of member-years had inpatient utilization?"
)

ToolRoute(tool=<ToolName.POPULATION_SUMMARY: 'get_population_summary'>, arguments={})

In [6]:
# Checkign for segment tool

route_question(
    "Describe the persistent high-cost complex segment."
)

ToolRoute(tool=<ToolName.SEGMENT_PROFILE: 'get_segment_profile'>, arguments={'segment': 'Persistent high-cost complex'})

In [7]:
# Checking for comparison tool

route_question(
    "Compare persistent high-cost complex members "
    "with pharmacy-intensive chronic members."
)

ToolRoute(tool=<ToolName.SEGMENT_COMPARISON: 'compare_segments'>, arguments={'segment_a': 'Persistent high-cost complex', 'segment_b': 'Pharmacy-intensive chronic'})

In [8]:
# Checkign for cost tool

route_question(
    "What share of positive healthcare spending "
    "comes from the top 10% of member-years?"
)

ToolRoute(tool=<ToolName.COST_CONCENTRATION: 'get_cost_concentration'>, arguments={'top_fraction': 0.1})

In [9]:
# Checking for condition tool

route_question(
    "Which chronic conditions are most common among "
    "recurrent high-cost complex members?"
)

ToolRoute(tool=<ToolName.CONDITION_PREVALENCE: 'get_condition_prevalence'>, arguments={'segment': 'Recurrent high-cost complex'})

All tools are being properly routed based on prompting.

In [3]:
from core.models import resolve_segment_name


print(
    resolve_segment_name(
        "Acute episodic high-cost"
    )
)

print(
    resolve_segment_name(
        "acute episodic high cost"
    )
)

print(
    resolve_segment_name(
        "  Acute   Episodic  High-Cost "
    )
)

SegmentName.ACUTE_EPISODIC
SegmentName.ACUTE_EPISODIC
SegmentName.ACUTE_EPISODIC


In [4]:
# Testing dispatcher agent

from agents.dispatcher import dispatch_tool
from agents.router import route_question

question = (
    "What share of positive spending "
    "comes from the top 10%?"
)

route = route_question(question)

result = dispatch_tool(route)

print(route)
result


tool=<ToolName.COST_CONCENTRATION: 'get_cost_concentration'> arguments={'top_fraction': 0.1}


{'top_fraction': 0.1,
 'top_percent': 10.0,
 'positive_member_years': 302102,
 'selected_member_years': 30210,
 'total_positive_spending': 1582713170.0,
 'selected_spending': 780230690.0,
 'spending_share_pct': 49.29703655653538}

In [5]:
# Test comparison

question = (
    "Compare persistent high-cost complex members "
    "with pharmacy-intensive chronic members."
)

route = route_question(question)

result = dispatch_tool(route)

print(route)
result["differences"]


tool=<ToolName.SEGMENT_COMPARISON: 'compare_segments'> arguments={'segment_a': 'Persistent high-cost complex', 'segment_b': 'Pharmacy-intensive chronic'}


{'mean_annual_cost': {'segment_a_value': 29925.566110397947,
  'segment_b_value': 7727.750024584521,
  'absolute_difference': 22197.816085813425,
  'higher_segment': 'Persistent high-cost complex'},
 'mean_conditions': {'segment_a_value': 6.486264441591785,
  'segment_b_value': 5.385878650801455,
  'absolute_difference': 1.1003857907903294,
  'higher_segment': 'Persistent high-cost complex'},
 'inpatient_prevalence_pct': {'segment_a_value': 71.96405648267009,
  'segment_b_value': 0.0,
  'absolute_difference': 71.96405648267009,
  'higher_segment': 'Persistent high-cost complex'},
 'mean_inpatient_days': {'segment_a_value': 9.242105263157894,
  'segment_b_value': 0.0,
  'absolute_difference': 9.242105263157894,
  'higher_segment': 'Persistent high-cost complex'},
 'mean_outpatient_claims': {'segment_a_value': 8.600513478818998,
  'segment_b_value': 5.132559740387452,
  'absolute_difference': 3.4679537384315458,
  'higher_segment': 'Persistent high-cost complex'},
 'mean_carrier_claims':

In [6]:
# Testing llm synthesizer

from agents.dispatcher import dispatch_tool
from agents.router import route_question
from agents.synthesizer import synthesize_answer


question = (
    "What share of positive healthcare spending "
    "comes from the top 10% of member-years?"
)

route = route_question(question)

tool_result = dispatch_tool(route)

answer = synthesize_answer(
    question=question,
    route=route,
    tool_result=tool_result,
)

print("ROUTE")
print(route)

print("\nTOOL RESULT")
print(tool_result)

print("\nANSWER")
print(answer)



ROUTE
tool=<ToolName.COST_CONCENTRATION: 'get_cost_concentration'> arguments={'top_fraction': 0.1}

TOOL RESULT
{'top_fraction': 0.1, 'top_percent': 10.0, 'positive_member_years': 302102, 'selected_member_years': 30210, 'total_positive_spending': 1582713170.0, 'selected_spending': 780230690.0, 'spending_share_pct': 49.29703655653538}

ANSWER
The top 10% of member-years (30,210 member-years) account for 49.3% of positive healthcare spending, based on synthetic CMS DE-SynPUF data. This represents the spending concentration for the highest-cost member-years in the dataset.


In [7]:
question = (
    "How do persistent high-cost complex members "
    "differ from pharmacy-intensive chronic members?"
)

route = route_question(question)

tool_result = dispatch_tool(route)

answer = synthesize_answer(
    question,
    route,
    tool_result,
)

print(answer)

The **Persistent high-cost complex** segment differs significantly from the **Pharmacy-intensive chronic** segment across key metrics, based on synthetic CMS DE-SynPUF data:  

1. **Cost and Utilization**:  
   - **Annual cost**: Persistent high-cost complex members had a mean annual cost of **$29,925.57** versus **$7,727.75** for pharmacy-intensive chronic members (difference: **$22,197.82**).  
   - **Inpatient care**: Persistent high-cost complex members had **71.96% inpatient prevalence** (mean inpatient days: **9.24**) versus **0%** for pharmacy-intensive chronic members.  
   - **Outpatient and carrier claims**: Persistent high-cost complex members had higher mean outpatient claims (**8.60** vs. **5.13**) and carrier claims (**47.75** vs. **29.40**).  

2. **Conditions and Complexity**:  
   - Persistent high-cost complex members had **6.49 mean conditions** versus **5.39** for pharmacy-intensive chronic members, indicating greater clinical complexity.  

3. **Pharmacy Use**:  
 

In [8]:
# Answer Synthesis evaluation

questions = [
    (
        "What percentage of member-years "
        "had inpatient utilization?"
    ),
    (
        "Describe persistent high-cost "
        "complex members."
    ),
    (
        "Compare persistent high-cost complex members "
        "with pharmacy-intensive chronic members."
    ),
    (
        "What share of positive spending "
        "comes from the top 10%?"
    ),
    (
        "Which chronic conditions are most common "
        "among recurrent high-cost complex members?"
    ),
]

for question in questions:

    route = route_question(question)

    result = dispatch_tool(route)

    answer = synthesize_answer(
        question,
        route,
        result,
    )

    print("=" * 80)
    print("QUESTION")
    print(question)

    print("\nROUTE")
    print(route)

    print("\nANSWER")
    print(answer)
    print()

QUESTION
What percentage of member-years had inpatient utilization?

ROUTE
tool=<ToolName.POPULATION_SUMMARY: 'get_population_summary'> arguments={}

ANSWER
The synthetic CMS DE-SynPUF data indicates that **13.5% of member-years** had inpatient utilization, as reflected by the `inpatient_prevalence_pct` metric. This percentage represents the proportion of member-years with at least one inpatient service during the study period (2008–2010). 

**Note:** This analysis uses synthetic CMS DE-SynPUF data and does not generalize to the real Medicare population.

QUESTION
Describe persistent high-cost complex members.

ROUTE
tool=<ToolName.SEGMENT_PROFILE: 'get_segment_profile'> arguments={'segment': 'Persistent high-cost complex'}

ANSWER
**Analytical Results:**  
The "Persistent high-cost complex" segment comprises **1,453 unique beneficiaries** with **3,895 member-years** of data. These members exhibit significantly elevated healthcare utilization and costs compared to typical Medicare bene